In [1]:
import json
import os
import warnings
from functools import reduce

import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", message=".*eval_set.*")   # LightGBM API-rename notice
# Loan amounts live on a ladder of standard values; used for the "within one step" metric
LADDER = np.array([5, 10, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 120, 150,
                   200, 250, 300, 400, 500, 800, 1000, 2000, 5000]) * 1000
csv = 'dataset/SingleCycleLoaneeMember.csv'
data = pd.read_csv(csv)

In [11]:
data['loan_amount'].to_numpy('float')[10]

np.float64(50000.0)

In [37]:
data.columns

Index(['member_id', 'name', 'dateofbirth', 'gender', 'current_addr_district',
       'current_addr_thana', 'maritial_status', 'admission_date', 'education',
       'occupation', 'alternative_occupation', 'total_family_member',
       'total_earning_member', 'total_monthly_income', 'yearly_total_income',
       'yearly_total_expenses', 'yearly_total_savings', 'home_owner_type',
       'year_of_rent', 'farm_land_amt', 'loan_id', 'proposer_oldid',
       'loan_cycle_no', 'loan_Component', 'loan_use_sources',
       'applied_loan_amount', 'loan_amount', 'installmenttype',
       'installmentyear', 'totalInstallmentcount'],
      dtype='object')

In [38]:
# Feature sets, smallest to largest. The ablation table shows what each addition buys.
FEATURE_SETS = {
    "1 income": ["monthly_income"],
    "2 +purpose": ["monthly_income", "purpose"],
    "3 +land": ["monthly_income", "purpose", "land"],
    "4 +family size": ["monthly_income", "purpose", "land", "family_size"],
    "5 +earners,expenses": ["monthly_income", "purpose", "land", "family_size",
                            "earners", "yearly_expenses"],
    "6 +district": ["monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district"],
    "7 +gender": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender"],
    "8 +occupation": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation" ],
    "9 +home_owner_type": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type" ],
    "10 +loan_Component": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component"],
    "11 +year_of_rent": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent"],
    "12 +installmenttype": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype"],
    "13 +installmentyear": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear"],
    "14 +totalInstallmentcount": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount"],
    "15 +maritial_status": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount", "maritial_status"],
    "16 +education": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount", "maritial_status", "education"],
    "17 +alternative_occupation": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount", "maritial_status", "education", "alternative_occupation"],
    "18 +admission_age": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                    "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount", "maritial_status",
                       "education", "alternative_occupation", "admission_age"],
}
DEFAULT_SET = "6 +district"
CAT_FEATURES = ["purpose", "district", 'gender','occupation','home_owner_type', 'loan_Component', 'installmenttype', 'maritial_status', 'education', 'alternative_occupation']

In [9]:
## work on purpose (EDA)
def create_purpose(s: pd.Series) -> pd.Series:
    return s.fillna('Na').astype(str).str.replace(r"\s*\(.*$", "", regex=True)

create_purpose(data['loan_use_sources'])


0                      Dairy Farm
1                Jute cultivation
2                      Dairy Farm
3              Banana Cultivation
4                      Dairy Farm
                   ...           
123988                 Dairy Farm
123989         Potato Cultivation
123990    Corn /Maize cultivation
123991          Tailor's Business
123992                    Fishery
Name: loan_use_sources, Length: 123993, dtype: object

In [40]:
# Feature building (same function for training and prediction)
def clean_purpose(s: pd.Series) -> pd.Series:
    """'Dairy Farm (ডেইরি ফার্ম)' -> 'Dairy Farm'"""
    return s.fillna("NA").astype(str).str.replace(r"\s*\(.*$", "", regex=True).str.strip()


def fit_meta(train_df: pd.DataFrame) -> dict:
    purpose = clean_purpose(train_df["loan_use_sources"])
    counts = purpose.value_counts()

    def categories(column):
        values = (
            train_df[column]
            .astype("string")
            .str.strip()
            .dropna()
            .unique()
            .tolist()
        )
        return sorted(values)

    return {
        "income_cap": float(
            train_df["total_monthly_income"].quantile(0.999)
        ),
        "expense_cap": float(
            train_df["yearly_total_expenses"].quantile(0.999)
        ),
        "land_sentinel": 1e5,
        "purposes": sorted(counts[counts >= 100].index.tolist()),
        "districts": categories("current_addr_district"),
        "gender": categories("gender"),
        "occupation": categories("occupation"),
        "home_owner_type": categories("home_owner_type"),
        "loan_Component": categories("loan_Component"),
        "installmenttype": categories("installmenttype"),
        "maritial_status": categories("maritial_status"),
        "education": categories("education"),
        "alternative_occupation": categories("alternative_occupation")
    }

In [116]:
data['alternative_occupation'].astype('string').str.strip().dropna().unique().tolist()

['Agriculture',
 'Business',
 'Bari bara',
 'No',
 'Krisi',
 'business',
 'Busniess',
 'Krisy',
 'Agricultural',
 'Agriculure',
 'Hand work',
 'KIRSE',
 'Job',
 'House Rent/ Job',
 'Probasi',
 'Farmer',
 'Dairy Farm',
 'অটোরিকশা',
 'Gavipalon',
 'Agreculture',
 'Immigrants',
 'Housewife',
 'farmar',
 'Fishery',
 'Agriculture. Katuri',
 'লাকড়ি ব্যবসা',
 'কৃষি',
 'হোটেল ব্যবসা',
 'Probashi',
 'agriculture',
 'BUSINESS',
 'Agriculer',
 'Farmar',
 'Krishi',
 'Zob',
 'Others',
 'housewife',
 'Service',
 'Basha bara',
 'Hospital Job',
 'Businesses',
 'Bisness',
 'Hash murgi palin',
 'Auto rikshaws',
 'Bussines',
 'Business + krishi + probashi',
 'Home Rent',
 'Dubai probashi',
 'Laber',
 'Fish buisness',
 'Driving',
 'Households',
 'Government  Job',
 'Day Labour',
 'Krise',
 'Buisness',
 'Pobasi',
 'Raj Mistry',
 'Drider',
 'Cow Business',
 'Cow',
 'Vegetable Business',
 'Cng driver',
 'Privat job',
 'Agri',
 'Agre',
 'Biasness',
 'Bessness',
 'Has murgi palon',
 'Sarvice',
 'Zg',
 'Trading

In [42]:
sorted(data['installmenttype'].unique().tolist())

['Mounthly', 'Weekly']

In [43]:
data['purpose']=clean_purpose(data["loan_use_sources"])
data['purpose']

0                      Dairy Farm
1                Jute cultivation
2                      Dairy Farm
3              Banana Cultivation
4                      Dairy Farm
                   ...           
123988                 Dairy Farm
123989         Potato Cultivation
123990    Corn /Maize cultivation
123991          Tailor's Business
123992                    Fishery
Name: purpose, Length: 123993, dtype: object

In [44]:
result=pd.crosstab(data['purpose'], data['gender'])
result['total'] = result.sum(axis=1)
result.sort_values('total', ascending=False)

gender,Female,Male,total
purpose,,,
Dairy Farm,29739,538,30277
Paddy Cultivation,20935,261,21196
Fishery,6006,251,6257
Auto Purchases,6005,210,6215
Immigrant Loan,5417,23,5440
...,...,...,...
Lime Making Business,1,0,1
Ladies Undergarments,0,1,1
Rice_Cooker Purchase,0,1,1


In [45]:
data.groupby(['gender', 'purpose']).agg(
    purposes=('purpose', 'count')
).sort_values('purposes', ascending=False)

purposes
gender purpose                                              
Female Dairy Farm                                      29739
       Paddy Cultivation                               20935
       Fishery                                          6006
       Auto Purchases                                   6005
       Immigrant Loan                                   5417
...                                                      ...
Male   Used Ship Equipment Business                        1
       Washing_Machine Purchase                            1
Female Gas Burner & Accessories Sell/Make Business         1
Male   Salt Cultivation                                    1
       Spice Cultivation                                   1

[458 rows x 1 columns]

In [46]:
fit_meta(data)

{'income_cap': 2000000.0,
 'expense_cap': 6000000.0,
 'land_sentinel': 100000.0,
 'purposes': ['Auto Purchases',
  'Auto battery purchase',
  'Bakery Business',
  'Banana Cultivation',
  'Battery Purchases',
  'Betel Leaf Business',
  'Betel Leaf Cultivation',
  'Bhangari Business',
  'Bicycle/Rickshaw/Van Making Business',
  'Bricks, Sand, Rod and Cement Business',
  'CNG Business',
  'Cloth Business',
  'Computer Business',
  'Confectionary',
  'Corn /Maize cultivation',
  'Cosmetic Business',
  'Crockery Business',
  'Dairy Farm',
  'Decorator Business',
  'Deep_Freezer / Refrigerator Purchase',
  'Distributor',
  'ELECTRIC & ELECTRONIC BUSSINES',
  'Electrical Equipment Making Business',
  'Electrical and Electronic Product Repair Business',
  'Engine Boat Rental Business',
  'Fertilizer, Pesticide and Seed Business',
  'Fish Business',
  'Fishery',
  'Fruit Business',
  'Furniture Purchase',
  'Garlic Cultivation',
  'Garment Business',
  'Goat Rearing',
  'Goru Motataja Koron',
 

In [132]:
def build_features(df: pd.DataFrame, meta: dict, features=None) -> pd.DataFrame:
    """Build only the requested features, so unused input columns are not required."""
    builders = {
        "monthly_income": lambda: df["total_monthly_income"].clip(upper=meta["income_cap"]),
        "purpose": lambda: pd.Categorical(
            (lambda p: p.where(p.isin(meta["purposes"]), "rare"))(clean_purpose(df["loan_use_sources"])),
            categories=meta["purposes"] + ["rare"]),                    # unseen/rare -> 'rare'
        "land": lambda: df["farm_land_amt"].where(df["farm_land_amt"] < meta["land_sentinel"]),
        "family_size": lambda: df["total_family_member"].clip(upper=12),
        "earners": lambda: df["total_earning_member"],
        'year_of_rent' : lambda: df["year_of_rent"],
        'installmentyear': lambda: df["installmentyear"],
        'totalInstallmentcount': lambda: df["totalInstallmentcount"],
        'admission_age' : lambda: (pd.to_datetime(df['admission_date']) - pd.to_datetime(df["dateofbirth"])).dt.days//365,
        "yearly_expenses": lambda: df["yearly_total_expenses"].clip(upper=meta["expense_cap"]),
        "district": lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["districts"])))(df["current_addr_district"].astype(str).str.strip()),
            categories=meta["districts"]),  # unseen -> missing
        'gender': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["gender"])))(df['gender'].astype(str))
        ),
        'occupation': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["occupation"])))(df['occupation'].astype(str))
        ),
        'home_owner_type': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["home_owner_type"])))(df['home_owner_type'].astype(str))
        ),
        'loan_Component': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["loan_Component"])))(df['loan_Component'].astype(str))
        ),
        'maritial_status': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["maritial_status"])))(df['maritial_status'].astype(str))
        ),
        'installmenttype': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["installmenttype"])))(df['installmenttype'].astype(str))
        ),
        'education': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["education"])))(df['education'].astype(str))
        ),
        'alternative_occupation' : lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["alternative_occupation"])))(df['alternative_occupation'].astype(str))
        ),

    }
    X = pd.DataFrame(index=df.index)
    for f in (features or list(builders)):
        X[f] = builders[f]()
    return X

In [133]:
pd.Categorical(data['gender'].astype('string').str.strip().dropna().unique().tolist())

['Female', 'Male']
Categories (2, object): ['Female', 'Male']

In [134]:
build_features(data, fit_meta(data))

,monthly_income,purpose,land,family_size,earners,year_of_rent,installmentyear,totalInstallmentcount,admission_age,yearly_expenses,district,gender,occupation,home_owner_type,loan_Component,maritial_status,installmenttype,education,alternative_occupation
0,100000.0,Dairy Farm,50.0,4,2,0.0,1.0,12,42,450000.0,Bhola,Female,Housewife,Own,JAGORON,Married,Mounthly,Secondary,NaN
1,40000.0,Jute cultivation,90.0,4,1,0.0,1.0,12,29,320000.0,Faridpur,Female,Housewife,Own,SUFOLON,Married,Mounthly,Pre-Primary,NaN
2,450000.0,Dairy Farm,10.0,5,1,0.0,1.0,12,37,400000.0,Shariatpur,Female,Housewife,Own,JAGORON,Married,Mounthly,Pre-Primary,NaN
3,80000.0,Banana Cultivation,20.0,4,2,0.0,1.0,46,44,500000.0,Meherpur,Female,Housewife,Own,JAGORON,Married,Weekly,Primary,NaN
4,30000.0,Dairy Farm,200.0,3,1,0.0,1.0,12,43,144000.0,Dinajpur,Male,Business,Own,JAGORON,Married,Mounthly,Pre-Primary,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123988,15000.0,Dairy Farm,0.0,4,2,0.0,1.0,46,18,90000.0,Dinajpur,Female,Housewife,Own,JAGORON,Married,Weekly,Secondary,Agriculture
123989,44000.0,Potato Cultivation,20.0,5,2,0.0,1.0,46,30,365000.0,Dinajpur,Female,Housewife,Own,SUFOLON,Married,Weekly,Primary,NaN
123990,42500.0,Corn /Maize cultivation,50.0,4,2,0.0,1.0,46,38,126000.0,Lalmonirhat,Female,Housewife,Own,JAGORON,Married,Weekly,PSC,NaN
123991,25000.0,Tailor's Business,0.0,4,2,0.0,1.0,46,18,100000.0,Chattogram,Female,Housewife,Own,JAGORON,Married,Weekly,Primary,Agriculture


In [135]:
# Metrics
def rung(v):
    return np.abs(np.log(np.asarray(v, float))[:, None] - np.log(LADDER)[None, :]).argmin(1)


def metrics(actual, pred) -> dict:
    a = np.asarray(actual, float)
    p = np.clip(np.asarray(pred, float), 4000, 5e6)
    la, lp = np.log1p(a), np.log1p(p)
    big = a >= 300000
    return {
        "MAE": float(np.mean(np.abs(p - a))),
        "MedianAE": float(np.median(np.abs(p - a))),
        "RMSLE": float(np.sqrt(np.mean((lp - la) ** 2))),
        "R2_log": float(1 - np.sum((la - lp) ** 2) / np.sum((la - la.mean()) ** 2)),
        "within_1_ladder_step": float(np.mean(np.abs(rung(p) - rung(a)) <= 1)),
        "exact_ladder_step": float(np.mean(rung(p) == rung(a))),
        "AUC_loan_ge_300k": float(roc_auc_score(big, np.log(p))) if 0 < big.sum() < len(a) else float("nan"),
    }


In [136]:
# Model
def make_model(n_estimators=3000):
    return lgb.LGBMRegressor(
        objective="l1",            # median of the conditional distribution; robust to the heavy tail
        learning_rate=0.05, num_leaves=15, min_child_samples=100,
        subsample=0.8, subsample_freq=1, reg_lambda=5,
        cat_smooth=30, min_data_per_group=50, cat_l2=10,
        n_estimators=n_estimators, verbose=-1, random_state=0,
    )


In [137]:
"""Early-stop on the most recent 15% of the training rows (time-ordered)."""
def fit_with_early_stopping(X, y_log, features):
    k = int(len(X) * 0.85)
    m = make_model()
    m.fit(X.iloc[:k][features], y_log[:k],
          eval_set=[(X.iloc[k:][features], y_log[k:])], eval_metric="l1",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m

In [29]:
a = [1, 2, 3]
b = [10, 20, 30]

result = list(map(lambda x, y: x + y, a, b))

print(result)
# [11, 22, 33]

[11, 22, 33]


In [60]:
np.full(4, 10)

array([10, 10, 10, 10])

In [138]:
def train(csv, outdir, feature_set=DEFAULT_SET, test_frac=0.2):
    os.makedirs(outdir, exist_ok=True)
    df = pd.read_csv(csv)
    df["_adm"] = pd.to_datetime(df["admission_date"])
    df = df.sort_values(["_adm", "member_id"]).reset_index(drop=True)   # time order
    y = df["loan_amount"].to_numpy(float)
    cut = int(len(df) * (1 - test_frac))
    tr, te = df.iloc[:cut], df.iloc[cut:]
    y_tr, y_te = y[:cut], y[cut:]
    print(f"rows {len(df):,} | train {len(tr):,} ({tr._adm.min().date()}..{tr._adm.max().date()}) "
          f"| test {len(te):,} ({te._adm.min().date()}..{te._adm.max().date()})")

    meta = fit_meta(tr)                         # constants from train only
    X_tr, X_te = build_features(tr, meta), build_features(te, meta)
    ly_tr = np.log1p(y_tr)

    # baselines
    results = {"global median": metrics(y_te, np.full(len(te), np.median(y_tr)))}    # know nothing benchmark
    bands = [-1, 30000, 40000, 50000, 60000, 70000, 100000, 1e12]
    key = lambda d, X: (X["purpose"].astype(str) + "|" +
                        pd.cut(d["total_monthly_income"], bands).astype(str))
    look = pd.Series(y_tr).groupby(key(tr, X_tr).values).median()
    fallback = pd.Series(y_tr).groupby(X_tr["purpose"].astype(str).values).median()
    pl = key(te, X_te).map(look).fillna(X_te["purpose"].astype(str).map(fallback)).fillna(np.median(y_tr))
    results["lookup: purpose x income band"] = metrics(y_te, pl.values)

    # feature-set ablation
    best_iters = {}
    for name, feats in FEATURE_SETS.items():
        m = fit_with_early_stopping(X_tr, ly_tr, feats)
        results[f"LightGBM [{name}]"] = metrics(y_te, np.expm1(m.predict(X_te[feats])))
        best_iters[name] = int(m.best_iteration_ or m.n_estimators)
        if name == feature_set:
            chosen_model = m
    table = pd.DataFrame(results).T
    pd.set_option("display.width", 200)
    print("\nTime-split test results (last %d%% of admissions)" % round(test_frac * 100))
    print(table.round(3).to_string())

    feats = FEATURE_SETS[feature_set]
    gain = pd.Series(chosen_model.booster_.feature_importance("gain"), index=feats)
    print("\nFeature importance (gain share) for", feature_set)
    print((gain / gain.sum()).sort_values(ascending=False).round(3).to_string())

    # refit on ALL rows with the chosen feature set
    meta_all = fit_meta(df)
    X_all = build_features(df, meta_all)
    final = make_model(n_estimators=int(best_iters[feature_set] * 1.1) + 1)
    final.fit(X_all[feats], np.log1p(y))
    final.booster_.save_model(os.path.join(outdir, "base_model.txt"))

    meta_all.update({"features": feats, "feature_set": feature_set,
                     "target_transform": "log1p", "n_estimators": final.n_estimators})
    with open(os.path.join(outdir, "base_model_meta.json"), "w") as f:
        json.dump(meta_all, f, indent=2)
    table.round(4).to_csv(os.path.join(outdir, "base_model_metrics.csv"))
    print(f"\nsaved base_model.txt, base_model_meta.json, base_model_metrics.csv -> {outdir}")

In [139]:
def predict(csv, outdir, out):
    with open(os.path.join(outdir, "base_model_meta.json")) as f:
        meta = json.load(f)
    booster = lgb.Booster(model_file=os.path.join(outdir, "base_model.txt"))
    df = pd.read_csv(csv)
    X = build_features(df, meta, meta["features"])
    pred = np.clip(np.expm1(booster.predict(X)), 4000, 5e6)
    res = pd.DataFrame({"predicted_loan": pred.round(0),
                        "nearest_standard_amount": LADDER[rung(pred)]})
    if "member_id" in df:
        res.insert(0, "member_id", df["member_id"].values)
    res.to_csv(out, index=False)
    print(f"wrote {len(res):,} predictions -> {out}")

In [140]:
outdir = "."
mode = 'train'
if mode == 'train':
    train(csv, outdir, DEFAULT_SET)
else:
    predict(csv,outdir,"predictions.csv" )

rows 123,993 | train 99,194 (2004-06-20..2025-06-01) | test 24,799 (2025-06-01..2026-07-24)

Time-split test results (last 20% of admissions)
                                             MAE   MedianAE  RMSLE  R2_log  within_1_ladder_step  exact_ladder_step  AUC_loan_ge_300k
global median                          58990.201  20000.000  0.765  -0.093                 0.357              0.084             0.500
lookup: purpose x income band          43123.432  20000.000  0.532   0.471                 0.448              0.229             0.926
LightGBM [1 income]                    45017.947  19984.229  0.551   0.433                 0.438              0.194             0.926
LightGBM [2 +purpose]                  42053.090  20000.009  0.514   0.506                 0.463              0.211             0.937
LightGBM [3 +land]                     41465.751  20000.009  0.506   0.521                 0.467              0.207             0.940
LightGBM [4 +family size]              41044.490  2000

In [84]:
# if __name__ == "__main__":
    # ap = argparse.ArgumentParser()
    # ap.add_argument("mode", choices=["train", "predict"])
    # ap.add_argument("--csv", required=True)
    # ap.add_argument("--outdir", default=".")
    # ap.add_argument("--feature-set", default=DEFAULT_SET, choices=list(FEATURE_SETS))
    # ap.add_argument("--out", default="predictions.csv")
    # a = ap.parse_args()
    # if a.mode == "train":
    #     train(a.csv, a.outdir, a.feature_set)
    # else:
    #     predict(a.csv, a.outdir, a.out)